# Hybrid RAG + Streamlit + Cloudflare Tunnel (Colab Optimized)

- OpenAI / Ollama 전환 가능
- Hybrid Search (BM25 + Dense) + Reranker
- Streamlit 웹 앱
- Cloudflare Tunnel로 외부 접속 가능
- GPU 메모리 최적화 적용

**사용 순서**
1. 런타임 → GPU 로 설정
2. 아래 셀을 위에서부터 순서대로 실행
3. 마지막 셀에서 나온 `trycloudflare.com` 주소로 접속

## 1. 패키지 설치

In [1]:
!pip install -U -q langchain langchain-community langchain-openai langchain-ollama langchain-chroma
!pip install -q chromadb rank_bm25 sentence-transformers pypdf
!pip install -q python-dotenv watchdog

# Cloudflare Tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

print("설치 완료")

(Reading database ... 127349 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.11.0) over (2026.11.0) ...
Setting up cloudflared (2026.11.0) ...
Processing triggers for man-db (2.12.0-4build2) ...
설치 완료


### 1-2. Ollama 설치 및 서버 실행
Ollama를 Colab 백그라운드에서 실행하고 필요한 모델들을 미리 다운로드합니다.

In [2]:
# 0. 필수 도구 zstd 설치 (Ollama 압축 해제용)
!sudo apt-get update -qq && sudo apt-get install -y -qq zstd

# 1. Ollama 설치
!curl -fsSL https://ollama.com/install.sh | sh

# 2. 백그라운드에서 Ollama 서버 백그라운드 실행
import subprocess
import time

print("Ollama 백그라운드 서버 시작 중...")
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5) # 서버가 정상 기동할 때까지 대기
print("Ollama 서버 실행 완료")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Ollama 백그라운드 서버 시작 중...
Ollama 서버 실행 완료


In [3]:
# 3. RAG에 필요한 LLM 모델 및 임베딩 모델 미리 Pull 받기
print("임베딩 모델(nomic-embed-text) 다운로드 중...")
!ollama pull nomic-embed-text

print("LLM 모델(llama3.2:3b) 다운로드 중...")
!ollama pull llama3.2:3b

print("모든 Ollama 모델 준비가 완료되었습니다!")

임베딩 모델(nomic-embed-text) 다운로드 중...

LLM 모델(llama3.2:3b) 다운로드 중...

모든 Ollama 모델 준비가 완료되었습니다!


## 2. GPU 메모리 관리 유틸리티

In [4]:
import torch
import gc
import os

def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        print(f"GPU 메모리 - Allocated: {allocated:.2f}GB / Reserved: {reserved:.2f}GB")
    else:
        print("GPU 사용 불가 (CPU 모드)")

clear_gpu()
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

GPU 메모리 - Allocated: 0.00GB / Reserved: 0.00GB
CUDA available: True
GPU Name: Tesla T4


## 3. Streamlit 앱 코드 생성 (app.py)

In [5]:
app_code = r'''
import streamlit as st
import torch
import gc
import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.retrievers import BM25Retriever

# Dynamic Import Fallbacks for Chains (history_aware, retrieval_chain, stuff_documents)
try:
    from langchain.chains import create_history_aware_retriever, create_retrieval_chain
    from langchain.chains.combine_documents import create_stuff_documents_chain
    HAS_OFFICIAL_CHAINS = True
except ImportError:
    try:
        from langchain.chains.history_aware_retriever import create_history_aware_retriever
        from langchain.chains.retrieval import create_retrieval_chain
        from langchain.chains.combine_documents import create_stuff_documents_chain
        HAS_OFFICIAL_CHAINS = True
    except ImportError:
        HAS_OFFICIAL_CHAINS = False

# Dynamic Import Fallbacks for EnsembleRetriever
try:
    from langchain.retrievers import EnsembleRetriever
except ImportError:
    try:
        from langchain_core.retrievers import EnsembleRetriever
    except ImportError:
        try:
            from langchain_community.retrievers.ensemble import EnsembleRetriever
        except ImportError:
            from langchain_core.retrievers import BaseRetriever
            from langchain_core.callbacks import CallbackManagerForRetrieverRun
            from langchain_core.documents import Document
            from typing import List
            class EnsembleRetriever(BaseRetriever):
                retrievers: List[BaseRetriever]
                weights: List[float]
                def _get_relevant_documents(self, query: str, *, run_manager: CallbackManagerForRetrieverRun) -> List[Document]:
                    all_docs = []
                    for r in self.retrievers:
                        try:
                            # Use invoke with standard config rather than passing callbacks directly
                            all_docs.extend(r.invoke(query))
                        except Exception:
                            try:
                                all_docs.extend(r.get_relevant_documents(query))
                            except Exception:
                                pass
                    return all_docs[:10]

# Dynamic Import Fallbacks for ContextualCompressionRetriever
try:
    from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
except ImportError:
    try:
        from langchain.retrievers import ContextualCompressionRetriever
    except ImportError:
        from langchain_core.vectorstores import VectorStoreRetriever
        ContextualCompressionRetriever = None

# Dynamic Import Fallbacks for CrossEncoderReranker
try:
    from langchain.retrievers.document_compressors import CrossEncoderReranker
except ImportError:
    try:
        from langchain_community.document_compressors import CrossEncoderReranker
    except ImportError:
        CrossEncoderReranker = None

from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from pypdf import PdfReader

load_dotenv()

st.set_page_config(page_title="Hybrid RAG (Colab Optimized)", page_icon="🚀", layout="wide")

# ======================
# 사이드바 설정
# ======================
with st.sidebar:
    st.title("설정")
    provider = st.selectbox("LLM Provider", ["openai", "ollama"], index=0)

    if provider == "openai":
        api_key = st.text_input("OpenAI API Key", type="password")
        model_name = st.selectbox("Model", ["gpt-4o-mini", "gpt-4o"])
    else:
        api_key = None
        # llama3.2:3b를 기본 선택되도록 맨 앞으로 위치 변경
        model_name = st.selectbox("Ollama Model", ["llama3.2:3b", "llama3.1:8b", "qwen2.5:7b"])

    use_reranker = st.checkbox("Reranker 사용", value=True)
    temperature = st.slider("Temperature", 0.0, 1.0, 0.1)

    st.markdown("---")
    if st.button("GPU 메모리 정리"):
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        st.success("메모리 정리 완료")

# ======================
# 모델 로딩 (캐싱)
# ======================
@st.cache_resource
def load_embeddings(provider, api_key=None):
    if provider == "ollama":
        return OllamaEmbeddings(model="nomic-embed-text")
    return OpenAIEmbeddings(model="text-embedding-3-small", api_key=api_key)

@st.cache_resource
def load_llm(provider, model_name, temperature, api_key=None):
    if provider == "ollama":
        return ChatOllama(model=model_name, temperature=temperature, num_ctx=4096)
    return ChatOpenAI(model=model_name, temperature=temperature, api_key=api_key, streaming=True)

@st.cache_resource
def load_reranker():
    return HuggingFaceCrossEncoder(
        model_name="BAAI/bge-reranker-base",
        device="cuda" if torch.cuda.is_available() else "cpu"
    )

# ======================
# 문서 처리
# ======================
st.header("Hybrid RAG Chat (Colab Optimized)")

uploaded_files = st.file_uploader("문서 업로드 (txt, pdf)", type=["txt", "pdf"], accept_multiple_files=True)

if "vectorstore" not in st.session_state:
    st.session_state.vectorstore = None
    st.session_state.documents = []

if uploaded_files:
    docs = []
    for file in uploaded_files:
        if file.name.endswith(".pdf"):
            pdf_reader = PdfReader(file)
            pdf_text = ""
            for page in pdf_reader.pages:
                page_text = page.extract_text()
                if page_text:
                    pdf_text += page_text + "\n"
            docs.append(pdf_text)
        else:
            content = file.read().decode("utf-8")
            docs.append(content)

    documents = [Document(page_content=d) for d in docs]

    splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=120)
    splits = splitter.split_documents(documents)
    st.session_state.documents = splits

    embeddings = load_embeddings(provider, api_key)
    st.session_state.vectorstore = Chroma.from_documents(splits, embeddings)
    st.success(f"{len(splits)}개 청크가 인덱싱되었습니다.")

# ======================
# Custom Chain Fallback (if LangChain chains are missing)
# ======================
class CustomRAGChain:
    def __init__(self, llm, retriever):
        self.llm = llm
        self.retriever = retriever

    def invoke(self, inputs):
        query = inputs["input"]
        chat_history = inputs.get("chat_history", [])

        # Optional: query contextualization
        context_query = query
        if chat_history:
            try:
                history_str = "\n".join([f"{m.type}: {m.content}" for m in chat_history])
                context_prompt = f"대화 기록:\n{history_str}\n\n위의 대화 기록을 기반으로 사용자의 다음 질문인 '{query}'를 독립적인 단일 질문으로 다시 정리해서 반환하세요."
                res = self.llm.invoke(context_prompt)
                context_query = res.content
            except Exception:
                pass

        try:
            docs = self.retriever.invoke(context_query)
        except Exception:
            try:
                docs = self.retriever.get_relevant_documents(context_query)
            except Exception:
                # Final safeguard fallback if retriever has custom implementations
                docs = []

        context_str = "\n\n".join([d.page_content for d in docs])

        system_msg = f"주어진 문서만을 근거로 답변하세요. 없는 내용은 추측하지 마세요.\n\n문서:\n{context_str}"
        prompt_messages = [("system", system_msg)]
        for msg in chat_history:
            prompt_messages.append((msg.type, msg.content))
        prompt_messages.append(("human", query))

        prompt_template = ChatPromptTemplate.from_messages(prompt_messages)
        formatted = prompt_template.format_messages()
        response = self.llm.invoke(formatted)

        return {
            "answer": response.content,
            "context": docs
        }

# ======================
# RAG 체인 생성
# ======================
def create_chain():
    if st.session_state.vectorstore is None:
        return None

    llm = load_llm(provider, model_name, temperature, api_key)

    dense_retriever = st.session_state.vectorstore.as_retriever(search_kwargs={"k": 8})

    bm25 = BM25Retriever.from_documents(st.session_state.documents)
    bm25.k = 8

    try:
        ensemble = EnsembleRetriever(retrievers=[bm25, dense_retriever], weights=[0.4, 0.6])
    except Exception:
        ensemble = dense_retriever

    if use_reranker and (CrossEncoderReranker is not None) and (ContextualCompressionRetriever is not None):
        try:
            reranker = CrossEncoderReranker(model=load_reranker(), top_n=3)
            retriever = ContextualCompressionRetriever(base_compressor=reranker, base_retriever=ensemble)
        except Exception:
            retriever = ensemble
    else:
        retriever = ensemble

    if HAS_OFFICIAL_CHAINS:
        try:
            contextualize_q_prompt = ChatPromptTemplate.from_messages([
                ("system", "대화 기록을 참고하여 질문을 독립적으로 재구성하세요."),
                MessagesPlaceholder("chat_history"),
                ("human", "{input}"),
            ])

            history_aware_retriever = create_history_aware_retriever(llm, retriever, contextualize_q_prompt)

            qa_prompt = ChatPromptTemplate.from_messages([
                ("system", "주어진 문서만을 근거로 답변하세요. 없는 내용은 추측하지 마세요.\n\n문서:\n{context}"),
                MessagesPlaceholder("chat_history"),
                ("human", "{input}"),
            ])

            question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
            return create_retrieval_chain(history_aware_retriever, question_answer_chain)
        except Exception:
            return CustomRAGChain(llm, retriever)
    else:
        return CustomRAGChain(llm, retriever)

# ======================
# 채팅 UI
# ======================
if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

if prompt := st.chat_input("질문을 입력하세요"):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    chain = create_chain()
    if chain is None:
        st.warning("먼저 문서를 업로드해주세요.")
    else:
        chat_history = []
        for m in st.session_state.messages[:-1]:
            if m["role"] == "user":
                chat_history.append(HumanMessage(content=m["content"]))
            else:
                chat_history.append(AIMessage(content=m["content"]))

        with st.chat_message("assistant"):
            with st.spinner("생각 중..."):
                result = chain.invoke({
                    "input": prompt,
                    "chat_history": chat_history
                })
                answer = result["answer"]
                st.markdown(answer)

                with st.expander("참고 문서"):
                    for i, doc in enumerate(result.get("context", [])):
                        st.markdown(f"**문서 {i+1}**")
                        st.write(doc.page_content[:400] + "...")

        st.session_state.messages.append({"role": "assistant", "content": answer})
'''

with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("✅ app.py 예외 복구 완료")

✅ app.py 예외 복구 완료


## 4. Streamlit 실행

In [6]:
import subprocess
import sys
import time

# 기존 프로세스 종료
!pkill -f streamlit || true
!pkill -f cloudflared || true
!pkill -f lt || true

# Streamlit 백그라운드 재실행
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true", "--server.enableCORS", "false"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(5)
print("✅ Streamlit 서버 재시작 완료 (port 8501)")

^C
^C
^C
✅ Streamlit 서버 재시작 완료 (port 8501)


## 5. Cloudflare Tunnel 실행 (외부 접속 URL 생성)

In [7]:
import subprocess
import time
import re

# 기존 터널 종료
!pkill -f cloudflared || true

# 터널 시작
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("Cloudflare Tunnel 시작 중... (최대 15초 대기)\n")

public_url = None
start_time = time.time()

while time.time() - start_time < 20:
    line = tunnel.stdout.readline()
    if not line:
        time.sleep(0.5)
        continue

    print(line.strip())

    # trycloudflare.com URL 추출
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print("\n" + "="*60)
    print("✅ 외부 접속 주소가 생성되었습니다!")
    print(f"🔗 {public_url}")
    print("="*60)
    print("\n위 주소를 브라우저에서 열어 Streamlit 앱을 사용하세요.")
else:
    print("\n⚠️ URL을 자동으로 찾지 못했습니다. 위 로그를 확인해주세요.")

^C
Cloudflare Tunnel 시작 중... (최대 15초 대기)

2026-11-17T06:44:23Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-11-17T06:44:23Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-11-17T06:44:26Z INF +--------------------------------------------------------------------------------------------+
2026-11-17T06:44:26Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-11-17T06:44:26Z INF |  https://hon

## 5-2. 대안 터널링 도구 실행 (localtunnel)
Cloudflare 속도가 느릴 경우, 대체제인 npm 기반의 `localtunnel`을 이용해 외부 접속 URL을 생성합니다.

In [8]:
# 1. nodejs 및 localtunnel 설치
!npm install -g localtunnel

# 2. 외부 접속용 IP 정보 확인 (localtunnel 접속 시 패스워드로 입력해야 합니다)
import urllib.request
public_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
print(f"🔑 localtunnel 접속 비밀번호 (Endpoint IP): {public_ip}")

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹
changed 22 packages in 1s
⠹
⠹3 packages are looking for funding
⠹  run `npm fund` for details
⠹🔑 localtunnel 접속 비밀번호 (Endpoint IP): 136.69.185.177


In [9]:
# 3. localtunnel 백그라운드 실행
import subprocess

# 기존 localtunnel 프로세스 종료
!pkill -f lt || true

lt_process = subprocess.Popen(
    ["lt", "--port", "8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

import time
time.sleep(5)
print("\nlocaltunnel에서 반환한 주소 확인 중...")
for _ in range(5):
    line = lt_process.stdout.readline()
    if "your url is" in line:
        print("\n" + "="*60)
        print("✅ 대체 우회 접속 주소가 생성되었습니다!")
        print(f"🔗 {line.strip()}")
        print(f"위 주소 접속 후 비밀번호 입력창에 [ {public_ip} ]를 입력하세요.")
        print("="*60)
        break
    time.sleep(1)

^C

localtunnel에서 반환한 주소 확인 중...

✅ 대체 우회 접속 주소가 생성되었습니다!
🔗 your url is: https://public-tigers-dream.loca.lt
위 주소 접속 후 비밀번호 입력창에 [ 136.69.185.177 ]를 입력하세요.


### 6. RAG 답변과 LLM 자체 지식 답변을 함께 제공하도록 app.py 업데이트

문서 기반 답변(RAG)뿐만 아니라 모델의 자체 지식 답변도 동시에 구해서 화면에 나란히 출력할 수 있도록 `app.py`의 구조를 고도화합니다.

In [10]:
app_code_dual = r'''
import streamlit as st
import torch
import gc
import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.retrievers import BM25Retriever

# Dynamic Import Fallbacks for Chains
try:
    from langchain.chains import create_history_aware_retriever, create_retrieval_chain
    from langchain.chains.combine_documents import create_stuff_documents_chain
    HAS_OFFICIAL_CHAINS = True
except ImportError:
    HAS_OFFICIAL_CHAINS = False

# Dynamic Import Fallbacks for EnsembleRetriever
try:
    from langchain.retrievers import EnsembleRetriever
except ImportError:
    try:
        from langchain_core.retrievers import EnsembleRetriever
    except ImportError:
        from langchain_core.retrievers import BaseRetriever
        from langchain_core.callbacks import CallbackManagerForRetrieverRun
        from langchain_core.documents import Document
        from typing import List
        class EnsembleRetriever(BaseRetriever):
            retrievers: List[BaseRetriever]
            weights: List[float]
            def _get_relevant_documents(self, query: str, *, run_manager: CallbackManagerForRetrieverRun) -> List[Document]:
                all_docs = []
                for r in self.retrievers:
                    try:
                        all_docs.extend(r.invoke(query))
                    except Exception:
                        pass
                return all_docs[:10]

from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, AIMessage
from pypdf import PdfReader

load_dotenv()

st.set_page_config(page_title="Hybrid RAG + LLM Comparison", page_icon="🚀", layout="wide")

# ======================
# 사이드바 설정
# ======================
with st.sidebar:
    st.title("설정")
    provider = st.selectbox("LLM Provider", ["openai", "ollama"], index=1)

    if provider == "openai":
        api_key = st.text_input("OpenAI API Key", type="password")
        model_name = st.selectbox("Model", ["gpt-4o-mini", "gpt-4o"])
    else:
        api_key = None
        model_name = st.selectbox("Ollama Model", ["llama3.2:3b", "llama3.1:8b", "qwen2.5:7b"])

    temperature = st.slider("Temperature", 0.0, 1.0, 0.3)

    st.markdown("---")
    if st.button("GPU 메모리 정리"):
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        st.success("메모리 정리 완료")

# ======================
# 모델 로딩 (캐싱)
# ======================
@st.cache_resource
def load_embeddings(provider, api_key=None):
    if provider == "ollama":
        return OllamaEmbeddings(model="nomic-embed-text")
    return OpenAIEmbeddings(model="text-embedding-3-small", api_key=api_key)

@st.cache_resource
def load_llm(provider, model_name, temperature, api_key=None):
    if provider == "ollama":
        return ChatOllama(model=model_name, temperature=temperature, num_ctx=4096)
    return ChatOpenAI(model=model_name, temperature=temperature, api_key=api_key, streaming=True)

# ======================
# 문서 처리
# ======================
st.header("RAG vs LLM 비교 분석 챗봇")
st.markdown("업로드된 문서 내용에만 기반한 대답과 LLM 자체의 지식을 사용한 대답을 함께 비교해 봅니다.")

uploaded_files = st.file_uploader("문서 업로드 (txt, pdf)", type=["txt", "pdf"], accept_multiple_files=True)

if "vectorstore" not in st.session_state:
    st.session_state.vectorstore = None
    st.session_state.documents = []

if uploaded_files:
    docs = []
    for file in uploaded_files:
        if file.name.endswith(".pdf"):
            pdf_reader = PdfReader(file)
            pdf_text = ""
            for page in pdf_reader.pages:
                page_text = page.extract_text()
                if page_text:
                    pdf_text += page_text + "\n"
            docs.append(pdf_text)
        else:
            content = file.read().decode("utf-8")
            docs.append(content)

    documents = [Document(page_content=d) for d in docs]
    splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=120)
    splits = splitter.split_documents(documents)
    st.session_state.documents = splits

    embeddings = load_embeddings(provider, api_key)
    st.session_state.vectorstore = Chroma.from_documents(splits, embeddings)
    st.success(f"{len(splits)}개 청크가 인덱싱되었습니다.")

# ======================
# 듀얼 답변 생성 클래스
# ======================
class DualResponseChain:
    def __init__(self, llm, retriever):
        self.llm = llm
        self.retriever = retriever

    def invoke(self, inputs):
        query = inputs["input"]
        chat_history = inputs.get("chat_history", [])

        # 1. 이전 대화를 기반으로 독립 질문 재구성 (역사 의식 검색)
        context_query = query
        if chat_history:
            try:
                history_str = "\n".join([f"{m.type}: {m.content}" for m in chat_history])
                context_prompt = f"대화 기록:\n{history_str}\n\n위의 대화 기록을 기반으로 사용자의 다음 질문인 '{query}'를 독립적인 단일 질문으로 다시 정리해서 반환하세요."
                res = self.llm.invoke(context_prompt)
                context_query = res.content
            except Exception:
                pass

        # 2. RAG용 문서 검색
        try:
            docs = self.retriever.invoke(context_query)
        except Exception:
            docs = []
        context_str = "\n\n".join([d.page_content for d in docs])

        # 3. RAG 답변 생성 (대화 기록 포함)
        rag_system = f"주어진 문서만을 근거로 구체적으로 답변하세요. 없는 내용은 절대 추측하거나 지어내지 마세요.\n\n[문서 내용]:\n{context_str}"
        rag_messages = [("system", rag_system)]
        for msg in chat_history:
            rag_messages.append((msg.type, msg.content))
        rag_messages.append(("human", query))

        rag_prompt = ChatPromptTemplate.from_messages(rag_messages)
        try:
            rag_response = self.llm.invoke(rag_prompt.format_messages())
            rag_answer = rag_response.content
        except Exception as e:
            rag_answer = f"RAG 답변 생성 중 오류가 발생했습니다: {str(e)}"

        # 4. LLM 자체 지식 답변 생성 (대화 기록 포함)
        llm_system = "당신은 유용한 AI 비서입니다. 당신이 기존에 학습하여 가진 자체 지식을 사용해 친절하고 자세히 답변해 주세요."
        llm_messages = [("system", llm_system)]
        for msg in chat_history:
            llm_messages.append((msg.type, msg.content))
        llm_messages.append(("human", query))

        llm_prompt = ChatPromptTemplate.from_messages(llm_messages)
        try:
            llm_response = self.llm.invoke(llm_prompt.format_messages())
            llm_answer = llm_response.content
        except Exception as e:
            llm_answer = f"LLM 답변 생성 중 오류가 발생했습니다: {str(e)}"

        return {
            "rag_answer": rag_answer,
            "llm_answer": llm_answer,
            "context": docs
        }

def create_dual_chain():
    if st.session_state.vectorstore is None:
        return None

    llm = load_llm(provider, model_name, temperature, api_key)
    dense_retriever = st.session_state.vectorstore.as_retriever(search_kwargs={"k": 5})

    try:
        bm25 = BM25Retriever.from_documents(st.session_state.documents)
        bm25.k = 5
        retriever = EnsembleRetriever(retrievers=[bm25, dense_retriever], weights=[0.4, 0.6])
    except Exception:
        retriever = dense_retriever

    return DualResponseChain(llm, retriever)

# ======================
# 채팅 UI
# ======================
if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        if msg["role"] == "user":
            st.markdown(msg["content"])
        else:
            col1, col2 = st.columns(2)
            with col1:
                st.info("**📄 업로드 문서 기반 답변 (RAG)**")
                st.markdown(msg["rag"])
            with col2:
                st.success("**🤖 모델 자체 지식 답변 (LLM)**")
                st.markdown(msg["llm"])

if prompt := st.chat_input("비교 질문을 입력하세요"):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    chain = create_dual_chain()
    if chain is None:
        st.warning("먼저 문서를 업로드해주세요.")
    else:
        chat_history = []
        for m in st.session_state.messages[:-1]:
            if m["role"] == "user":
                chat_history.append(HumanMessage(content=m["content"]))
            elif m["role"] == "assistant" and "rag" in m:
                # RAG와 LLM 대화 중 맥락 유지를 위해 RAG 답변 내용을 임시 대화 조각으로 활용
                chat_history.append(AIMessage(content=m["rag"]))

        with st.chat_message("assistant"):
            with st.spinner("답변 생성 중..."):
                res = chain.invoke({"input": prompt, "chat_history": chat_history})

                col1, col2 = st.columns(2)
                with col1:
                    st.info("**📄 업로드 문서 기반 답변 (RAG)**")
                    st.markdown(res["rag_answer"])
                with col2:
                    st.success("**🤖 모델 자체 지식 답변 (LLM)**")
                    st.markdown(res["llm_answer"])

                with st.expander("참조한 문서 소스 보기"):
                    for i, doc in enumerate(res.get("context", [])):
                        st.markdown(f"**문서 조각 {i+1}**")
                        st.write(doc.page_content)

        st.session_state.messages.append({
            "role": "assistant",
            "content": "Dual Response",
            "rag": res["rag_answer"],
            "llm": res["llm_answer"]
        })
'''

with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code_dual)

print("✅ 대화 내용을 정상 기억하는 듀얼 비교 app.py 완성 및 갱신 완료!")

✅ 대화 내용을 정상 기억하는 듀얼 비교 app.py 완성 및 갱신 완료!


In [11]:
import subprocess
import time

# 최신 듀얼 코드(app_code_dual)를 다시 한번 확실히 app.py에 기록
with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code_dual)
print("📝 app.py 파일이 최종 듀얼 모드 코드로 업데이트되었습니다.")

# 기존 Streamlit 프로세스 종료
!pkill -f streamlit || true

# Streamlit 재시작
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true", "--server.enableCORS", "false"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(3)
print("✅ Streamlit 서버가 대화 맥락 유지 기능이 포함된 새로운 듀얼 비교 모드로 재시작되었습니다. (포트: 8501)")

📝 app.py 파일이 최종 듀얼 모드 코드로 업데이트되었습니다.
^C
✅ Streamlit 서버가 대화 맥락 유지 기능이 포함된 새로운 듀얼 비교 모드로 재시작되었습니다. (포트: 8501)


## 사용 방법

1. 런타임을 **GPU**로 설정하세요.
2. 위 셀들을 순서대로 실행합니다.
3. 마지막 셀에서 출력된 `https://xxxx.trycloudflare.com` 주소로 접속합니다.
4. 사이드바에서 Provider를 선택하고 (OpenAI 권장), API Key를 입력합니다.
5. txt 문서를 업로드한 후 질문을 입력하면 됩니다.

> **참고**: Colab 세션이 끊기면 Tunnel도 함께 종료됩니다. 시현할 때마다 다시 실행해야 합니다.